# Read me

In this notebook, we use a Autoencoder with a NearestNeighbors model with Euclidean distance.

9 features → Encoder → 4 embeddings → k-NN → 20 recommandations

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

from keras import Sequential, Input, Model
from keras.layers import Dense
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping

from IPython.display import HTML, display

import joblib

# Import Spotify module
import app.spotify.spotify as spo

In [3]:
# features
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

# hyper parameters
latent_dimension = 4
input_dimension = 9
hidden_dimension = 16

# Spotify access token
token = spo.get_access_token()


# Load data

In [4]:
df = pd.read_csv("../raw_data/spotify-tracks-dataset-detailed.csv")
print(f'Loaded {len(df)} rows')

Loaded 114000 rows


# Pre-process data

1. Remove Comedy
2. Drop duplicates
3. Drop missing values

In [5]:
len_before = len(df)
df = df[df["track_genre"] != "comedy"].reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} comedy tracks. Left {len_after} rows')

len_before = len(df)
df = df.drop_duplicates(subset=["track_id"]).drop_duplicates(subset=["artists", "track_name"]).reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} duplicates. Left {len_after} rows')

len_before = len(df)
df = df.dropna(subset=['album_name']).dropna(subset=['track_name']).reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} null values. Left {len(df)} rows')

Removed 1000 comedy tracks. Left 113000 rows
Removed 32642 duplicates. Left 80358 rows
Removed 1 null values. Left 80357 rows


# Split Train Val Test

In [6]:
X = df[audio_features]

X_train, X_temp = train_test_split(
    X, test_size=0.30, random_state=42
)

X_val, X_test = train_test_split(
    X_temp, test_size=0.50, random_state=42
)

# Scaling

In [7]:
scaler = StandardScaler().set_output(transform="pandas")

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Build autoencoder
1. Build encoder
2. Build decoder
3. Stack them together

In [8]:
def build_encoder(input_dimension, hidden_dimension, latent_dimension):
    """
    Returns an encoder model
    """

    encoder = Sequential(name="encoder")

    encoder.add(Input(shape=(input_dimension,)))
    encoder.add(Dense(hidden_dimension, activation='relu'))
    encoder.add(Dense(latent_dimension, activation='linear'))

    return encoder


def build_decoder(latent_dimension, hidden_dimension, input_dimension=9):
    """
    Returns an decoder model
    """

    decoder = Sequential(name="decoder")

    decoder.add(Input(shape=(latent_dimension,)))
    decoder.add(Dense(hidden_dimension, activation="relu"))
    decoder.add(Dense(input_dimension, activation="linear"))

    return decoder


encoder = build_encoder(input_dimension, hidden_dimension, latent_dimension)
decoder = build_decoder(latent_dimension, hidden_dimension, input_dimension)

inputs = Input(shape=(9,))
outputs = decoder(encoder(inputs))

autoencoder = Model(inputs, outputs)

autoencoder.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="mse"
)

autoencoder.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 9)]               0         
                                                                 
 encoder (Sequential)        (None, 4)                 228       
                                                                 
 decoder (Sequential)        (None, 9)                 233       
                                                                 
Total params: 461
Trainable params: 461
Non-trainable params: 0
_________________________________________________________________


# Train

In [9]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_train_scaled,
    X_train_scaled,
    validation_data=(
        X_val_scaled,
        X_val_scaled
    ),
    epochs=100,
    batch_size=256,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100


2026-11-07 18:02:38.118471: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


220/220 [==============================] - 1s 543us/step - loss: 0.7197 - val_loss: 0.4531
Epoch 2/100
220/220 [==============================] - 0s 287us/step - loss: 0.4014 - val_loss: 0.3445
Epoch 3/100
220/220 [==============================] - 0s 284us/step - loss: 0.2807 - val_loss: 0.2282
Epoch 4/100
220/220 [==============================] - 0s 284us/step - loss: 0.2090 - val_loss: 0.1902
Epoch 5/100
220/220 [==============================] - 0s 282us/step - loss: 0.1813 - val_loss: 0.1710
Epoch 6/100
220/220 [==============================] - 0s 285us/step - loss: 0.1665 - val_loss: 0.1615
Epoch 7/100
220/220 [==============================] - 0s 287us/step - loss: 0.1596 - val_loss: 0.1567
Epoch 8/100
220/220 [==============================] - 0s 283us/step - loss: 0.1554 - val_loss: 0.1535
Epoch 9/100
220/220 [==============================] - 0s 284us/step - loss: 0.1523 - val_loss: 0.1504
Epoch 10/100
220/220 [==============================] - 0s 281us/step - loss: 0.1493 

In [14]:
# save encoder
encoder.save('../artifacts/autoencoder_knn/autoencoder_encoder.keras')

# Encode X into embedings

In [10]:
# Encode the tracks

X_scaled = scaler.transform(X)
X_embeddings = encoder.predict(X_scaled, verbose=0)

print(X_scaled.shape)
print(X_embeddings.shape)

(80357, 9)
(80357, 4)


# Train k-NN model with embedings

In [11]:
# NearestNeighbors with Euclidean distance
model = NearestNeighbors(metric="euclidean")

model.fit(X_embeddings)

NearestNeighbors(metric='euclidean')

In [13]:
# save model

joblib.dump(model, "../artifacts/autoencoder_knn/autoencoder_knn_model.joblib")

['../artifacts/autoencoder_knn/autoencoder_knn_model.joblib']

# Build input tracks list

In [4]:
# Build an input track set of 10 tracks
genre = "jazz"

input_tracks = df[df["track_genre"] == genre].sample(
    n=10,
    random_state=42
)

In [ ]:
# input_tracks.to_csv('jazz.csv', index=False)

In [44]:
input_tracks[['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]

,track_id,artists,album_name,track_name,track_genre
48644,0Oho1Elt6nNyiHZyZR8OaG,Jane Monheit,Surrender,A Time For Love,jazz
48653,7esh5OJ7CP435rmXeGm305,Eartha Kitt,Classic Christmas Greatest Hits,Santa Baby,jazz
48577,5cQANJMHoDKb2fCxr5pZkq,Leo Gandelman;Baco Exu do Blues;Machine Series,Hip Hop Machine #6,Flamingos,jazz
48616,4nbILrYODP667b3d66ZMzI,Dean Martin,"Hey, Brother Pour The Wine",Sway (Quien Sera),jazz
48454,1fBUmT2qNGdTgaS4XXwGnA,Nat King Cole,Christmas Only: The Timeless Classic Collection,O Little Town of Bethlehem,jazz
48550,13cpOnmccZGVGFngLxYN4I,Kenny G,Breathless,Forever in Love,jazz
48621,4m6OC8u2rC6LMFtFWsME7d,Chet Baker,Chet Baker Sings,Time After Time - Vocal Version,jazz
48512,6SvIO9X1alZpKyG3g8EAHB,Peggy Lee,Christmas Classics,The Christmas Waltz,jazz
48645,3Nsn3y2IpzcuDIKZDgS3oS,Luis Arcaraz Y Su Orquesta,Cita Con Luis Arcaraz,Viajera,jazz
48578,4v5PKC5rvQwb33NWX82UH8,Carlos Lyra,Bossa Nova Carlos Lyra,Você e Eu (You and Me),jazz


In [40]:
# Get input track indices
input_indices = input_tracks.index.to_list()

In [37]:
playlist_id = spo.create_playlist(token)
spo.add_tracks_to_playlist(token, playlist_id, df.iloc[input_indices]['track_id'])
spo.display_player(playlist_id)

# Recommend

In [45]:
# Calculate input profile
input_profile = X_embeddings[input_indices].mean(axis=0).reshape(1, -1)

# Find 20 neighbors
distances, indices = model.kneighbors(input_profile, n_neighbors=20)

In [ ]:
# Show player
playlist_id = spo.create_playlist(token)
spo.add_tracks_to_playlist(token, playlist_id, df.iloc[indices[0]]['track_id'])
spo.display_player(playlist_id)

In [43]:
# The model returns an array of indices
# Return the tracks from the original dataframe
recommendations = df.iloc[indices[0]]
recommendations[['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]


,track_id,artists,album_name,track_name,track_genre
42892,0xHvmWCpz9Pnn1sDvRehAs,Pritam;Arijit Singh,Ae Dil Hai Mushkil (Original Motion Picture So...,Channa Mereya - Unplugged,indian
66100,24fsdEtEb9U7cwlPvbc5xr,Nikolay Kopylov,"Ради Женщин (Сцены, Арии И Дуэты Из Классическ...",Ария Раджами Из Оперетты «Баядера»,romance
26760,0VxocBntP1XTZRsR9ZURPS,Daniela Andrade,La Vie En Rose,La Vie En Rose,folk
10321,7jFNX9aTdcQfhIgUVE02hi,Hacken Lee,命運符號,月半小夜曲,cantopop
65228,4rkv812lQ9utiRQ9EMLIKb,The Platters,The Very Best Of The Platters,I'm Sorry,rockabilly
438,68KCwQ2PNKNT17sEx32DEG,The Civil Wars,Barton Hollow,Dance Me to the End of Love - Bonus,acoustic
13309,3r9eGg4SVPbhtiPTA9UkmK,keshi,always,us,chill
58775,7jKCAmjP2cAn9WxxIVR14Y,Skylar Grey,Aquaman (Original Motion Picture Soundtrack) [...,Everything I Need,piano
3369,2kNwTOVdZGCGROgAzjD1J0,Julien Baker,Guthrie,Guthrie,ambient
10415,1SjCVgrZqywAgwe0U8pGJk,Eason Chan,The Line-Up,人來人往,cantopop
